# 10 FRP 조화: 불 세기도 MODIS 기준으로

03의 모델은 칸 × 월의 MODIS **탐지 수**를 맞힙니다. 여기서는 특징, Tweedie 목적함수, 단조 제약을 그대로 두고 맞힐 대상만 칸 × 월의 **MODIS FRP 합계(MW)** 로 바꿉니다 (`fsml/frp.py`).

- 2013–2019년 전 세계 합으로 보면 탐지 수는 VIIRS가 MODIS의 약 4.4배인데, FRP 합계는 1.03배입니다. VIIRS는 화소가 작아 같은 불을 여러 개로 나눠 세기 때문입니다.
- 하지만 칸 × 월로 보면 VIIRS FRP 합계가 MODIS의 2배 이내인 경우가 54%뿐이라, VIIRS FRP를 그대로 쓸 수는 없습니다.
- FRP는 단위가 MW라서 탐지 수와 더하거나 같은 축에 놓지 않습니다.

## 미리 정한 기준 (결과를 보기 전에 정함)

**시험 방법:** 05·09와 같은 연도별 시험입니다. 2020–2024년의 시험 연도 y마다 y−2년까지 학습하고 y−1년으로 조기 종료합니다. 학습에는 Suomi NPP 입력만 씁니다.

**비교 대상(기준선)** (규칙 모델은 탐지 수 전용이라 FRP에는 쓰지 않습니다):
- **비율 기준선:** 위도대(10°) × 월별 MODIS FRP ÷ VIIRS FRP 비율을 VIIRS FRP에 곱합니다 (02의 기준선과 같은 방식).
- **VIIRS FRP 그대로:** VIIRS FRP 합계를 바꾸지 않고 씁니다.

**판정 기준:**
1. **FRP 모델 채택** (셋 다 만족해야 함)
   - 합계 MAE(10° 블록 × 월)의 5년 평균이 두 기준선 중 나은 쪽보다 **10% 이상** 작음
   - **5년 중 4년 이상** 그 기준선보다 합계 MAE가 작음
   - 평균 |편향|이 그 기준선보다 **5%p 이하**로만 큼
2. **2026년(NOAA-20)에도 적용** (둘 다 만족해야 함)
   - NOAA-20 입력의 합계 MAE 평균이 NPP 입력의 **1.10배 이하**
   - NOAA-20 입력이 기준선(NOAA-20 입력)보다 **5년 중 4년 이상** 오차가 작음
3. **80% 범위 공개:** 03과 같은 분할(2022년 조기 종료·보정, 2023–2024년 시험)에서 포함률이 **75–85%**

**미달일 때:**
- 1이 미달이면 `harmonized_frp`를 만들지 않습니다.
- 2가 미달이면 2026년 FRP는 비워 둡니다.
- 3이 미달이면 `frp_low` / `frp_high`를 비워 둡니다.

판정은 `models/frp/decision.json`에 남기고, `python main.py frp`가 이를 따릅니다.

In [ ]:
import json
import time

import numpy as np
import pandas as pd

from fsml import evaluate, frp, gbm  # evaluate: 한글 폰트 설정 포함
from fsml.baseline import RatioBaseline
from fsml.compare import PRIMARY, _scores
from fsml.features import make_features, training_rows
from fsml.intervals import ConformalInterval, contain, coverage
from fsml.paths import grid_path
from fsml.splits import time_folds, time_split

RES = 0.5
TEST_YEARS = [2020, 2021, 2022, 2023, 2024]
T = frp.TARGET
BASELINES = {
    "비율 기준선": lambda: RatioBaseline(target=frp.TARGET, source=frp.SOURCE),
    "VIIRS FRP 그대로": None,
}

grid_npp = pd.read_parquet(grid_path(RES, "month"))
grid_n20 = pd.read_parquet(grid_path(RES, "month", "_noaa20"))
npp, FEATURES = make_features(training_rows(grid_npp), RES)
n20, features_n20 = make_features(training_rows(grid_n20), RES)
assert FEATURES == features_n20
yr_npp = pd.to_datetime(npp["period"]).dt.year.to_numpy()
yr_n20 = pd.to_datetime(n20["period"]).dt.year.to_numpy()
print(f"NPP 행 {len(npp):,} | NOAA-20 행 {len(n20):,} | 특징 {len(FEATURES)}개")

## 1. 원래 값: VIIRS ÷ MODIS (연도별, 전 세계)

In [ ]:
def yearly(d):
    g = d.groupby(pd.to_datetime(d["period"]).dt.year)[["modis_n", "viirs_n", T, frp.SOURCE]].sum()
    return pd.DataFrame({"탐지 수 VIIRS÷MODIS": g["viirs_n"] / g["modis_n"],
                         "FRP VIIRS÷MODIS": g[frp.SOURCE] / g[T]})

pd.concat({"NPP": yearly(npp), "NOAA-20": yearly(n20)}, axis=1).round(2)

## 2. 연도별 시험

In [ ]:
rows = []
t0 = time.time()
for (tr, va, te), y in zip(time_folds(npp, TEST_YEARS), TEST_YEARS):
    model = gbm.fit(npp, FEATURES, tr, va, log_every=0, target=T)
    base = BASELINES["비율 기준선"]().fit(npp[tr | va])
    for name, d, mask in (("NPP", npp, te), ("NOAA-20", n20, yr_n20 == y)):
        t = d[mask]
        preds = {"LightGBM": model.predict(t[FEATURES]), "비율 기준선": base.predict(t),
                 "VIIRS FRP 그대로": t[frp.SOURCE].to_numpy(float)}
        for m, p in preds.items():
            rows.append({"연도": y, "입력": name, "모델": m, **_scores(t, p, 10, target=T)})
results = pd.DataFrame(rows)
print(f"완료 ({time.time() - t0:.0f}초)")
wide = results.pivot_table(index="연도", columns=["입력", "모델"], values=PRIMARY)
wide.loc["평균"] = wide.mean()
wide.round(0)

In [ ]:
results.groupby(["입력", "모델"])[[PRIMARY, "칸 MAE", "칸 R²(log)", "편향 %"]].mean().round(3)

## 3. 판정 (미리 정한 기준 1, 2)

In [ ]:
def table(inp):
    return results[results["입력"] == inp].pivot(index="연도", columns="모델", values=PRIMARY)

def abs_bias(inp):
    r = results[results["입력"] == inp]
    return r.assign(b=r["편향 %"].abs()).groupby("모델")["b"].mean()

w = table("NPP")
best = min(BASELINES, key=lambda m: w[m].mean())
improvement = float(1 - w["LightGBM"].mean() / w[best].mean())
wins = int((w["LightGBM"] < w[best]).sum())
extra_bias = float(abs_bias("NPP")["LightGBM"] - abs_bias("NPP")[best])
c1 = {"합계 MAE 10% 이상 개선": improvement >= 0.10, "5년 중 4년 이상 승리": wins >= 4,
      "편향 증가 5%p 이하": extra_bias <= 5.0}

w20 = table("NOAA-20")
best20 = min(BASELINES, key=lambda m: w20[m].mean())
ratio20 = float(w20["LightGBM"].mean() / w["LightGBM"].mean())
wins20 = int((w20["LightGBM"] < w20[best20]).sum())
c2 = {"NOAA-20 합계 MAE ≤ NPP의 1.10배": ratio20 <= 1.10, "NOAA-20이 기준선보다 4년 이상 나음": wins20 >= 4}

print(f"기준선 중 나은 쪽: {best} | 개선 {improvement:+.1%} | 승리 {wins}/5 | |편향| 차이 {extra_bias:+.1f}%p")
print(f"NOAA-20: 합계 MAE NOAA-20 ÷ NPP {ratio20:.3f} | 기준선({best20})보다 나은 해 {wins20}/5")
pd.Series({**{f"1. {k}": v for k, v in c1.items()}, **{f"2. {k}": v for k, v in c2.items()}}).map({True: "통과", False: "미달"})

## 4. 80% 범위 (미리 정한 기준 3)

03과 같은 분할입니다: 2021년까지 학습, 2022년의 절반으로 조기 종료하고 나머지 절반으로 보정, 2023–2024년으로 시험합니다.

In [ ]:
train, valid, test = time_split(npp)
model_t = gbm.fit(npp, FEATURES, train, valid, log_every=0, target=T)
interval_t = ConformalInterval(coverage=0.8, target=T).fit(npp, FEATURES, train, valid)
X = npp.loc[test, FEATURES]
low, high = contain(model_t.predict(X), *interval_t.predict(X))
cov = coverage(npp.loc[test, T], low, high)
c3 = {"포함률 75–85%": 0.75 <= cov["포함률"] <= 0.85}
print(f"보정 여유 {interval_t.margin:.1f} MW")
pd.Series({**{k: round(v, 3) for k, v in cov.items()}, "판정": "통과" if c3["포함률 75–85%"] else "미달"})

## 5. 판정 저장, 최종 모델 학습

In [ ]:
verdict = {"adopt": all(c1.values()), "noaa20": all(c1.values()) and all(c2.values()),
           "range": all(c1.values()) and all(c3.values())}
frp.FOLDER.mkdir(parents=True, exist_ok=True)
(frp.FOLDER / "decision.json").write_text(json.dumps(verdict, indent=1))
if verdict["adopt"]:
    final, final_interval = frp.fit_final(grid_npp, RES)
    print(f"최종 모델 저장: {frp.FOLDER} (트리 {final.num_trees()}개, 범위 보정 여유 {final_interval.margin:.1f} MW)")
verdict